In [0]:
from pyspark.sql import functions as f
from datetime import datetime
from delta.tables import DeltaTable
import uuid

In [0]:
spark.sql("""
          use catalog ecommerce_data
          """)

In [0]:
spark.sql("""
          create schema if not exists bronze
          """)

Step 1: Create Bronze control table to help the pipeline with:
1. latest timestamp processed
2. latest primary key processed for the timestamp
3. How many rows were written

In [0]:
spark.sql("""
          create or replace table ecommerce_data.bronze.ingestion_control(
              layer string,
              table_name string,
              ts_col string,
              pk_col string,
              last_successfull_ts timestamp,
              last_successfull_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at timestamp
          )
          using delta
          """)

Step 2 : create table config,
It'll let us know, for each source table load which column should be used as ts_col and pk_col

In [0]:
table_config={
              'orders':{'pk_col':'order_id','ts_col':'updated_at'},
              'payments':{'pk_col':'payment_id','ts_col':'processed_at'},
              'products':{'pk_col':'product_id','ts_col':'updated_at'}
          }

bronze_run_id=str(uuid.uuid4())

print("Current Bronze Run Id:",bronze_run_id)

Step 3: Create two helper functions
1 -> To get the last successfull watermark
2 -> To update the ingestion_control table after the successfull bronze load

1 -> get_last_successfull_watermark()

filter the rows using ctrl and get the latest timestamp,

if filter is none then it's the new table 
else return the values

In [0]:
def get_last_successfull_watermark(table_name:str):
    ctrl=(
        spark.table("ecommerce_data.bronze.ingestion_control").filter(
        (f.col('layer')=='Bronze') &
        (f.col('table_name')==table_name) &
        (f.col('run_status')=='success')
    )
    .orderBy(f.col('updated_at').desc())
    .limit(1)
    )
    rows=ctrl.collect()

    if not rows:
        return None, None
    return rows[0]['last_successfull_ts'],rows[0]['last_successfull_pk']    

2nd function - This function will update the ingestion control table after the succesfull bronze load

In [0]:
def upsert_bronze_control(
    table_name,
    ts_col,
    pk_col,
    last_ts,
    last_pk,
    rows_written,
    run_id
):
    df_control = spark.createDataFrame(
        [(
            'Bronze',
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            str(run_id),
            int(rows_written),
            'success',
            datetime.utcnow()
        )],
        schema="""
            layer STRING,
            table_name STRING,
            ts_col STRING,
            pk_col STRING,
            last_successfull_ts TIMESTAMP,
            last_successfull_pk BIGINT,
            last_run_id STRING,
            rows_written BIGINT,
            run_status STRING,
            updated_at TIMESTAMP
        """
    )

    dt = DeltaTable.forName(
        spark,
        "ecommerce_data.bronze.ingestion_control"
    )

    (
        dt.alias("t")
        .merge(
            df_control.alias("s"),
            """
            t.table_name = s.table_name
            AND t.layer = s.layer
            """
        )
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successfull_ts": "s.last_successfull_ts",
            "last_successfull_pk": "s.last_successfull_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

Step 4-
Bronze layer ingestion loop
1. fetch latest watermark
2. read the source table
3. filter new/updated rows
4. add audit column
5. update the bronze detla table
6. update control table

In [0]:

for table_name, cols in table_config.items():
    pk_col = cols["pk_col"]
    ts_col = cols["ts_col"]

    source_table = f"`ecommerce_sql_connection_catalog`.dbo.{table_name}"
    target_table = f"ecommerce_data.bronze.{table_name}_raw"

    last_successfull_ts, last_successfull_pk = get_last_successfull_watermark(table_name)

    print(f"\n== Processing {table_name} ==")
    print(f"Last Successful Timestamp: {last_successfull_ts}")
    print(f"Last Successful PK: {last_successfull_pk}")

    source_df = (
        spark.read.table(source_table)
        .withColumn(ts_col, f.col(ts_col).cast("timestamp"))
    )

    # Identify new records using the composite watermark
    if last_successfull_ts is None:
        new_df = source_df
    else:
        new_df = source_df.filter(
            (f.col(ts_col) > f.lit(last_successfull_ts))
            | (
                (f.col(ts_col) == f.lit(last_successfull_ts))
                & (
                    f.col(pk_col).cast("long")
                    > f.lit(int(last_successfull_pk))
                )
            )
        )

    # Add Bronze ingestion metadata
    new_df = (
        new_df
        .withColumn("bronze_ingested_at", f.current_timestamp())
        .withColumn("bronze_run_id", f.lit(bronze_run_id))
        .withColumn("source_table", f.lit(source_table))
    )

    rows_count = new_df.count()
    print(f"{table_name} - New Records: {rows_count}")

    if rows_count == 0:
        print(f"No new records found for {table_name}")

        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successfull_ts,
            last_successfull_pk,
            rows_count,
            bronze_run_id
        )

        continue

    # Append new records to the Bronze Delta table
    (
        new_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )

    # Calculate the new composite watermark
    max_ts = new_df.agg(
        f.max(ts_col).alias("max_ts")
    ).collect()[0]["max_ts"]

    max_pk = (
        new_df
        .filter(f.col(ts_col) == f.lit(max_ts))
        .agg(f.max(f.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    # Update the control table after a successful write
    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id
    )

    print(f"Wrote {rows_count} records to {target_table}")


Validating our tables

In [0]:
print('Orders rows count:',spark.sql('select count(*) from ecommerce_data.bronze.orders_raw').collect()[0][0])
print('Payments rows count:',spark.sql('select count(*) from ecommerce_data.bronze.payments_raw').collect()[0][0])
print('products rows count:',spark.sql('select count(*) from ecommerce_data.bronze.products_raw').collect()[0][0])
display(spark.sql('select * from ecommerce_data.bronze.ingestion_control order by table_name'))